# QREV v4.0.0 reviewed analytical preflight

**Family:** Reverberation / observable residual-tail manifestations  
**Candidate only:** this notebook does not run the frozen cohort and cannot authorize publication freeze.

This notebook applies the common G1–G10 validation architecture used for QGAIN and QADD. The critical reviewed correction is that boundary-dependent QREV measurements use frozen `primary_speech / primary` offsets. Frozen `strict_speech / primary` is an already-eroded support view and is used only to quantify speech support for SRMR; it is never substituted for the natural speech offset.

Permitted claim: observable post-offset residual magnitude, bounded persistence, conditional downward decay, and a pinned reverberation-sensitive modulation comparator. Prohibited claims include RT60, EDT, C50/C80, D50, DRR, STI, recovered RIR, or causal echo identity.

In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys, tempfile, wave
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.io import loadmat, wavfile


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src reviewed').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Open this notebook from the paper1 project root.')

ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src reviewed'))
sys.path.insert(0, str(ROOT / 'src'))
from paper1_qc_reviewed.qrev_v400 import *

RUN_PACKAGE_TESTS = True
RUN_SRMR_RUNTIME = True
RUN_CODEC_ROUNDTRIP = True
RUN_COHORT_EXTRACTION = False
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = 'PENDING'

OUT = ROOT / 'outputs reviewed' / 'reverberation' / 'qrev-v4.0.0-candidate'
TABLES = OUT / 'tables'
FIGURES = OUT / 'figures'
AUDIT = OUT / 'audit'
MANIFESTS = OUT / 'manifests'
for folder in [TABLES, FIGURES, AUDIT, MANIFESTS]:
    folder.mkdir(parents=True, exist_ok=True)

PARAMETER_HASH = hashlib.sha256(json.dumps(DEFAULT_PARAMETERS.to_dict(), sort_keys=True).encode()).hexdigest()
IMPLEMENTATION_PATH = ROOT / 'src reviewed' / 'paper1_qc_reviewed' / 'qrev_v400.py'
IMPLEMENTATION_HASH = hashlib.sha256(IMPLEMENTATION_PATH.read_bytes()).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def save_df(frame, stem, folder=TABLES):
    path = folder / f'{stem}.csv'
    frame.to_csv(path, index=False)
    return path


def save_figure(fig, stem, source, caption, panel, scientific_question):
    fig.tight_layout()
    png = FIGURES / f'{stem}.png'
    svg = FIGURES / f'{stem}.svg'
    pdf = FIGURES / f'{stem}.pdf'
    csv = FIGURES / f'{stem}.source.csv'
    cap = FIGURES / f'{stem}.caption.md'
    prov = FIGURES / f'{stem}.provenance.json'
    fig.savefig(png, dpi=300, bbox_inches='tight')
    fig.savefig(svg, bbox_inches='tight')
    fig.savefig(pdf, bbox_inches='tight')
    source.to_csv(csv, index=False)
    cap.write_text(caption, encoding='utf-8')
    provenance = {
        'panel': panel,
        'scientific_question': scientific_question,
        'measurement_version': MEASUREMENT_VERSION,
        'parameter_sha256': PARAMETER_HASH,
        'implementation_sha256': IMPLEMENTATION_HASH,
        'source_csv': csv.name,
        'source_csv_sha256': sha256_file(csv),
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'feature_values_recomputed_from_cohort': False,
    }
    prov.write_text(json.dumps(provenance, indent=2), encoding='utf-8')
    return {'panel': panel, 'figure_id': stem, 'png': png.name, 'svg': svg.name, 'pdf': pdf.name, 'source': csv.name, 'caption': cap.name, 'provenance': prov.name}

checks = []
fig_index = []

def check(gate, item, passed, observed, required, note=''):
    checks.append({'gate': gate, 'check': item, 'passed': bool(passed), 'observed': str(observed), 'required': str(required), 'note': note})

print('Project root:', ROOT)
print('Candidate output:', OUT)

In [ ]:
# G1 — scientific contract, registry, and corrected boundary provenance
registry = feature_registry_frame()
parameters = pd.DataFrame([DEFAULT_PARAMETERS.to_dict()])

check('G1', 'exact four-feature vector', tuple(registry.feature) == ANALYSIS_FEATURES, tuple(registry.feature), ANALYSIS_FEATURES)
check('G1', 'natural primary-speech boundaries required', (registry.boundary_view.iloc[:3] == 'primary_speech').all(), registry.boundary_view.iloc[:3].tolist(), 'primary_speech for all boundary estimators')
check('G1', 'strict speech is support-only for SRMR', registry.boundary_view.iloc[3] == 'primary task span; strict speech support', registry.boundary_view.iloc[3], 'separate span/support views')
check('G1', 'SRMR implementation fully pinned', SRMR_UPSTREAM_COMMIT and SRMR_GAMMATONE_VERSION == '1.0.3' and DEFAULT_PARAMETERS.srmr_max_modulation_cf_hz == 30.0, f'{SRMR_VARIANT}; {SRMR_UPSTREAM_COMMIT}; Gammatone {SRMR_GAMMATONE_VERSION}', 'pinned variant, commit, dependencies, filterbank and bandwidth')
check('G1', 'no scalar or standalone threshold', registry.family_scalar_prohibited.all() and registry.standalone_gate_prohibited.all(), 'prohibited', 'prohibited')

save_df(registry, 'qrev_v400_feature_registry')
save_df(parameters, 'qrev_v400_parameters')
registry

In [ ]:
# Shared synthetic fixtures and perturbation helpers
FS = 16_000

def synthetic_task(fs=FS, speech_count=6, speech_sec=.8, pause_sec=1.2, tail_level=0.0, tau=.12, plateau=None, floor_level=2e-5, seed=12, deterministic=False):
    rng = np.random.default_rng(seed)
    total = speech_count * speech_sec + (speech_count - 1) * pause_sec
    y = np.zeros(round(total * fs), dtype=float)
    primary, strict = [], []
    cursor = 0.0
    for i in range(speech_count):
        start, end = cursor, cursor + speech_sec
        primary.append(SpeechInterval(start, end, f'p{i}', i, 'primary_speech', 'primary'))
        strict.append(SpeechInterval(start + .05, end - .05, f's{i}', i, 'strict_speech', 'primary'))
        left, right = round(start * fs), round(end * fs)
        t = np.arange(right - left) / fs
        y[left:right] += .04 * (np.sin(2*np.pi*173*t) + .5*np.sin(2*np.pi*421*t+.3) + .2*np.sin(2*np.pi*911*t+.7))
        if i < speech_count - 1 and tail_level > 0:
            pause_right = round((end + pause_sec) * fs)
            rel = np.arange(pause_right - right) / fs
            carrier = (np.sin(2*np.pi*257*rel) + .35*np.sin(2*np.pi*619*rel+.2)) if deterministic else rng.standard_normal(len(rel))
            env = tail_level * (rel < plateau) if plateau is not None else tail_level * np.exp(-rel/tau)
            y[right:pause_right] += env * carrier
        cursor = end + (pause_sec if i < speech_count - 1 else 0.0)
    t = np.arange(len(y)) / fs
    y += floor_level * (np.sin(2*np.pi*997*t) + .5*np.sin(2*np.pi*1301*t+.1)) if deterministic else floor_level * rng.standard_normal(len(y))
    return y, primary, strict


def ext(y, primary, strict, *, compute_srmr=False, parameters=DEFAULT_PARAMETERS, recording_id='synthetic'):
    return extract_qrev(y, FS, primary_speech=primary, strict_speech=strict, logical_recording_id=recording_id, compute_srmr=compute_srmr, parameters=parameters)


def feature_srmr(waveform):
    """QREV feature-level SRMR after the declared global DC-removal step."""
    return compute_srmr_norm(remove_global_dc(waveform), FS)


def synthetic_rir(rt60_sec, fs=FS, seed=440):
    if rt60_sec <= 0:
        return np.array([1.0])
    rng = np.random.default_rng(seed)
    length = max(2, round((rt60_sec + .15) * fs))
    t = np.arange(length) / fs
    tau = rt60_sec / np.log(1000.0)  # amplitude reaches -60 dB at rt60
    h = np.zeros(length)
    h[0] = 1.0
    h += .08 * rng.standard_normal(length) * np.exp(-t/tau)
    return h / np.sqrt(np.sum(h*h))


def apply_rir(y, rt60_sec):
    h = synthetic_rir(rt60_sec)
    return signal.fftconvolve(y, h)[:len(y)]


def add_post_offset(y, primary, kind, amp=.01, seed=7):
    out = y.copy(); rng = np.random.default_rng(seed)
    for boundary in internal_pause_boundaries(primary, len(y)/FS):
        left = round(boundary['speech_offset_sec'] * FS)
        pause_n = round(min(1.0, boundary['pause_duration_sec']) * FS)
        effect_n = round(min(.55, boundary['pause_duration_sec']) * FS)
        t = np.arange(effect_n) / FS
        if kind == 'stationary':
            # Same additive process spans both early and independent late-floor windows.
            out[left:left+pause_n] += amp * rng.standard_normal(pause_n)
        elif kind == 'late_floor_step':
            floor_left = left + round(.70 * FS); floor_right = min(left + round(1.0 * FS), left+pause_n)
            out[floor_left:floor_right] += amp * rng.standard_normal(max(0, floor_right-floor_left))
        elif kind == 'breath':
            out[left:left+effect_n] += amp * np.exp(-t/.18) * rng.standard_normal(effect_n)
        elif kind == 'echo':
            delay = round(.14 * FS); width = round(.12 * FS)
            source = out[max(0,left-width):left].copy()
            destination = left + delay
            out[destination:destination+len(source)] += .7 * source
        elif kind == 'rising':
            out[left:left+effect_n] += amp * (t/max(t.max(),1e-12)) * rng.standard_normal(effect_n)
    return out


def align_length(values, target):
    values = np.asarray(values, dtype=float)
    if len(values) < target:
        values = np.pad(values, (0, target-len(values)))
    return values[:target]


def codec_roundtrip(y, codec):
    ffmpeg = shutil.which('ffmpeg')
    if not ffmpeg:
        raise RuntimeError('ffmpeg unavailable')
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        source = td / 'source.wav'
        wavfile.write(source, FS, np.asarray(np.clip(y, -1, 1) * 32767, dtype=np.int16))
        if codec == 'flac':
            encoded = td / 'encoded.flac'; command = [ffmpeg, '-y', '-v', 'error', '-i', str(source), str(encoded)]
        elif codec == 'opus':
            encoded = td / 'encoded.opus'; command = [ffmpeg, '-y', '-v', 'error', '-i', str(source), '-c:a', 'libopus', '-b:a', '64k', str(encoded)]
        elif codec == 'aac':
            encoded = td / 'encoded.m4a'; command = [ffmpeg, '-y', '-v', 'error', '-i', str(source), '-c:a', 'aac', '-b:a', '96k', str(encoded)]
        else:
            raise ValueError(codec)
        subprocess.run(command, check=True)
        decoded = td / 'decoded.wav'
        subprocess.run([ffmpeg, '-y', '-v', 'error', '-i', str(encoded), '-ac', '1', '-ar', str(FS), '-c:a', 'pcm_s16le', str(decoded)], check=True)
        rate, values = wavfile.read(decoded)
        values = values.astype(float) / (32768.0 if values.dtype.kind == 'i' else 1.0)
        return align_length(values, len(y))

base, primary, strict = synthetic_task(tail_level=.01, plateau=.18, deterministic=True)
boundaries = internal_pause_boundaries(primary, len(base)/FS)
check('G1', 'natural offset is 50 ms later than already-eroded strict end', abs(boundaries[0]['speech_offset_sec']-primary[0].end_sec)<1e-12 and abs(boundaries[0]['speech_offset_sec']-strict[0].end_sec-.05)<1e-12, {'primary_end': primary[0].end_sec, 'strict_end': strict[0].end_sec}, 'primary end is boundary; strict end differs by -50 ms')

In [ ]:
# G2 — numerical correctness, reconstruction, determinism, and pinned SRMR regression
first = ext(base, primary, strict)
second = ext(base, primary, strict)
check('G2', 'deterministic extraction', first.recording == second.recording and first.boundary_ledger.equals(second.boundary_ledger), 'exact repeat', 'exact repeat')

ledger_map = {
    'qrev_tail_excess_100ms_db': ('tail_eligible', 'tail_excess_100ms_db'),
    'qrev_tail_persistence_median_sec': ('persistence_eligible', 'tail_persistence_sec'),
    'qrev_downward_decay_rate_db_per_sec': ('decay_eligible', 'downward_decay_rate_db_per_sec'),
}
for feature, (eligible, value) in ledger_map.items():
    local = first.boundary_ledger.loc[first.boundary_ledger[eligible].astype(bool), value]
    if np.isfinite(first.recording[feature]):
        reconstructed = float(np.median(local))
        check('G2', f'{feature} reconstructs exactly from boundary ledger', abs(reconstructed-first.recording[feature])<1e-10, reconstructed, first.recording[feature])

negative_floor, p0, s0 = synthetic_task(floor_level=1e-5, deterministic=True)
rng = np.random.default_rng(4)
for boundary in internal_pause_boundaries(p0, len(negative_floor)/FS):
    left = round((boundary['speech_offset_sec']+.70)*FS); right = round((boundary['speech_offset_sec']+1.0)*FS)
    negative_floor[left:right] += .003*rng.standard_normal(right-left)
negative_result = ext(negative_floor, p0, s0).recording
check('G2', 'signed tail excess is not clipped at zero', negative_result[ANALYSIS_FEATURES[0]] < 0, negative_result[ANALYSIS_FEATURES[0]], '<0 under elevated independent late floor')

nondecay, pn, sn = synthetic_task(tail_level=.01, plateau=1.0, deterministic=True)
nondecay_result = ext(nondecay, pn, sn).recording
check('G2', 'nondecaying trace is unavailable rather than zero decay', np.isnan(nondecay_result[ANALYSIS_FEATURES[2]]) and nondecay_result[f'{ANALYSIS_FEATURES[2]}_status']=='no_valid_downward_decay', nondecay_result[f'{ANALYSIS_FEATURES[2]}_status'], 'no_valid_downward_decay')

srmr_runtime_available = False
srmr_reference = pd.DataFrame()
if RUN_SRMR_RUNTIME:
    try:
        fixture = ROOT / 'tests' / 'fixtures' / 'srmrpy' / 'test.mat'
        sample = loadmat(fixture)['s'][:,0]
        installed = importlib.metadata.version('Gammatone')
        observed = compute_srmr_norm(sample, FS)
        srmr_runtime_available = True
        srmr_reference = pd.DataFrame([{
            'variant': SRMR_VARIANT,
            'upstream_commit': SRMR_UPSTREAM_COMMIT,
            'gammatone_version': installed,
            'observed': observed,
            'expected': SRMR_PINNED_REGRESSION_VALUE,
            'absolute_error': abs(observed-SRMR_PINNED_REGRESSION_VALUE),
        }])
        check('G2', 'pinned SRMR regression fixture', installed==SRMR_GAMMATONE_VERSION and abs(observed-SRMR_PINNED_REGRESSION_VALUE)<1e-9, observed, SRMR_PINNED_REGRESSION_VALUE)
    except Exception as exc:
        check('G2', 'pinned SRMR regression fixture', False, repr(exc), 'Gammatone 1.0.3 and exact SRMRpy fixture result')
else:
    check('G2', 'pinned SRMR regression fixture', False, 'NOT RUN', 'must run locally before preflight acceptance')
if len(srmr_reference):
    save_df(srmr_reference, 'qrev_v400_srmr_reference_audit')

dc_contract_source = sample if srmr_runtime_available else base
dc_contract_reference = remove_global_dc(dc_contract_source)
dc_contract_shifted = remove_global_dc(dc_contract_source + .25)
check(
    'G2',
    'declared global DC-removal preprocessing is exact',
    np.max(np.abs(dc_contract_reference-dc_contract_shifted)) < 1e-12
    and abs(float(np.mean(dc_contract_shifted))) < 1e-12,
    {
        'max_absolute_difference': float(np.max(np.abs(dc_contract_reference-dc_contract_shifted))),
        'post_removal_mean': float(np.mean(dc_contract_shifted)),
    },
    'constant DC offsets removed before QREV feature extraction',
)


In [ ]:
# G3 — transformation, source-rate, codec, and duration behavior
baseline = ext(base, primary, strict).recording
transformation_rows = []
conditions = {
    'baseline': (base, primary, strict),
    'gain_plus_9_db': (apply_gain_db(base, 9), primary, strict),
    'polarity': (-base, primary, strict),
    'dc_plus_0.25': (base+.25, primary, strict),
}
shift_sec = .37; shift_n = round(shift_sec*FS)
conditions['common_time_shift'] = (np.pad(base, (shift_n,0)), shift_intervals(primary,shift_sec), shift_intervals(strict,shift_sec))
for label, (waveform, pp, ss) in conditions.items():
    result = ext(waveform, pp, ss, recording_id=label).recording
    transformation_rows.append({'condition': label, **{feature: result[feature] for feature in ANALYSIS_FEATURES[:3]}})
transform = pd.DataFrame(transformation_rows)
for feature in ANALYSIS_FEATURES[:3]:
    delta = np.nanmax(np.abs(transform[feature]-transform.loc[transform.condition=='baseline',feature].iloc[0]))
    check('G3', f'{feature} gain/polarity/DC/common-shift invariant', delta<1e-8, delta, '<1e-8')

# Relevant resampling path: synthesize the same continuous signal at 16 and 48 kHz, then downsample 48 -> 16 once.
y16, p16, s16 = synthetic_task(fs=16000, tail_level=.012, tau=.15, deterministic=True)
y48, _, _ = synthetic_task(fs=48000, tail_level=.012, tau=.15, deterministic=True)
y48_to_16 = align_length(signal.resample_poly(y48, 1, 3), len(y16))
r16 = ext(y16, p16, s16).recording
r48 = ext(y48_to_16, p16, s16).recording
resampling = pd.DataFrame([{
    'source_rate_hz': 48000,
    'analysis_rate_hz': 16000,
    **{f'{feature}_reference': r16[feature] for feature in ANALYSIS_FEATURES[:3]},
    **{f'{feature}_resampled': r48[feature] for feature in ANALYSIS_FEATURES[:3]},
    **{f'{feature}_absolute_delta': abs(r48[feature]-r16[feature]) if np.isfinite(r48[feature]) and np.isfinite(r16[feature]) else np.nan for feature in ANALYSIS_FEATURES[:3]},
}])
check('G3', 'tail excess stable under deterministic source-rate resampling', resampling[f'{ANALYSIS_FEATURES[0]}_absolute_delta'].iloc[0] <= .15, resampling[f'{ANALYSIS_FEATURES[0]}_absolute_delta'].iloc[0], '<=0.15 dB')
check('G3', 'persistence stable under deterministic source-rate resampling', resampling[f'{ANALYSIS_FEATURES[1]}_absolute_delta'].iloc[0] <= .02, resampling[f'{ANALYSIS_FEATURES[1]}_absolute_delta'].iloc[0], '<=0.02 s')
decay_relative = abs(r48[ANALYSIS_FEATURES[2]]-r16[ANALYSIS_FEATURES[2]])/r16[ANALYSIS_FEATURES[2]] if np.isfinite(r16[ANALYSIS_FEATURES[2]]) and np.isfinite(r48[ANALYSIS_FEATURES[2]]) else np.nan
check('G3', 'conditional decay eligibility and source-rate stability', np.isfinite(decay_relative) and decay_relative<=.12, decay_relative, '<=12% relative error')

codec_rows=[]
if RUN_CODEC_ROUNDTRIP:
    # Use an above-quantization synthetic floor for the codec characterization. The
    # lower-floor analytical fixture is appropriate for floating-point estimator tests
    # but would be erased by 16-bit PCM staging before codec encoding.
    codec_base, codec_primary, codec_strict = synthetic_task(
        tail_level=.012,
        tau=.15,
        plateau=.18,
        floor_level=2e-4,
        deterministic=True,
        seed=26,
    )
    codec_reference = ext(
        codec_base,
        codec_primary,
        codec_strict,
        recording_id='codec_reference',
    ).recording
    codec_rows.append({
        'codec':'unencoded_reference',
        'status':'measured',
        **{feature:codec_reference[feature] for feature in ANALYSIS_FEATURES[:3]},
        **{f'{feature}_status':codec_reference[f'{feature}_status'] for feature in ANALYSIS_FEATURES[:3]},
        **{f'{feature}_available':bool(np.isfinite(codec_reference[feature])) for feature in ANALYSIS_FEATURES[:3]},
        **{f'{feature}_absolute_delta':0.0 for feature in ANALYSIS_FEATURES[:3]},
    })
    for codec in ['flac','opus','aac']:
        try:
            decoded=codec_roundtrip(codec_base,codec)
            result=ext(decoded,codec_primary,codec_strict,recording_id=codec).recording
            codec_rows.append({
                'codec':codec,
                'status':'measured',
                **{feature:result[feature] for feature in ANALYSIS_FEATURES[:3]},
                **{f'{feature}_status':result[f'{feature}_status'] for feature in ANALYSIS_FEATURES[:3]},
                **{f'{feature}_available':bool(np.isfinite(result[feature])) for feature in ANALYSIS_FEATURES[:3]},
                **{f'{feature}_absolute_delta':abs(result[feature]-codec_reference[feature]) if np.isfinite(result[feature]) and np.isfinite(codec_reference[feature]) else np.nan for feature in ANALYSIS_FEATURES[:3]},
            })
        except Exception as exc:
            codec_rows.append({'codec':codec,'status':f'error:{type(exc).__name__}:{exc}'})
codec_table=pd.DataFrame(codec_rows)
encoded_codec_table=codec_table.loc[codec_table.codec.ne('unencoded_reference')]
check(
    'G3',
    'codec behavior characterized without assuming invariance',
    len(encoded_codec_table)==3 and encoded_codec_table.status.eq('measured').all(),
    encoded_codec_table.status.tolist() if len(encoded_codec_table) else [],
    'FLAC, Opus, and AAC measured against an above-quantization reference fixture',
)
check(
    'G3',
    'codec characterization preserves feature availability',
    all(encoded_codec_table.get(f'{feature}_available',pd.Series(dtype=bool)).all() for feature in ANALYSIS_FEATURES[:3]),
    {feature:encoded_codec_table.get(f'{feature}_available',pd.Series(dtype=bool)).tolist() for feature in ANALYSIS_FEATURES[:3]},
    'all three conditional tail features available in every codec condition',
)

srmr_transform=pd.DataFrame(); srmr_duration=pd.DataFrame()
if srmr_runtime_available:
    fixture_sample=loadmat(ROOT/'tests'/'fixtures'/'srmrpy'/'test.mat')['s'][:,0]
    base_score=feature_srmr(fixture_sample)
    srows=[]
    for label,waveform in [('baseline',fixture_sample),('gain_plus_9_db',apply_gain_db(fixture_sample,9)),('polarity',-fixture_sample),('dc_plus_0.25',fixture_sample+.25)]:
        score=feature_srmr(waveform); srows.append({'condition':label,'qrev_srmr_norm':score,'absolute_delta':abs(score-base_score),'relative_delta':abs(score-base_score)/base_score})
    srmr_transform=pd.DataFrame(srows)
    check('G3','QREV SRMR pipeline gain/polarity/DC behavior',srmr_transform.relative_delta.max()<=.01,srmr_transform.relative_delta.max(),'<=1% relative difference after declared global DC removal')
    drows=[]
    for repetitions in [1,2,3]:
        score=feature_srmr(np.tile(fixture_sample,repetitions)); drows.append({'repetitions':repetitions,'duration_sec':len(fixture_sample)*repetitions/FS,'qrev_srmr_norm':score,'relative_delta_from_one':abs(score-base_score)/base_score})
    srmr_duration=pd.DataFrame(drows)
    check('G3','pinned SRMR repeated-content duration stability',srmr_duration.relative_delta_from_one.max()<=.15,srmr_duration.relative_delta_from_one.max(),'<=15% relative difference')

save_df(transform,'qrev_v400_transformation_controls')
save_df(resampling,'qrev_v400_source_rate_resampling')
save_df(codec_table,'qrev_v400_codec_characterization')
if len(srmr_transform): save_df(srmr_transform,'qrev_v400_srmr_transformation_controls')
if len(srmr_duration): save_df(srmr_duration,'qrev_v400_srmr_duration_stability')

# Panel C — exact invariance, resampling tolerances, and codec characterization
# are shown separately so that codec reporting scales are not mistaken for
# analytical acceptance tolerances.
tolerance={'tail':.15,'persistence':.02,'decay_relative':.12}
resampling_plot=pd.DataFrame([
    {'condition':'48k->16k','feature':'tail excess','normalized_deviation':resampling[f'{ANALYSIS_FEATURES[0]}_absolute_delta'].iloc[0]/tolerance['tail']},
    {'condition':'48k->16k','feature':'persistence','normalized_deviation':resampling[f'{ANALYSIS_FEATURES[1]}_absolute_delta'].iloc[0]/tolerance['persistence']},
    {'condition':'48k->16k','feature':'decay','normalized_deviation':decay_relative/tolerance['decay_relative']},
])
codec_reporting_scale={'tail excess':1.0,'persistence':.02,'decay':.10}
codec_plot_rows=[]
for _,row in encoded_codec_table.loc[encoded_codec_table.status.eq('measured')].iterrows():
    decay_ref=codec_reference[ANALYSIS_FEATURES[2]]
    decay_rel=(row[f'{ANALYSIS_FEATURES[2]}_absolute_delta']/abs(decay_ref)) if np.isfinite(decay_ref) and decay_ref!=0 else np.nan
    codec_plot_rows += [
        {'condition':row.codec,'feature':'tail excess','absolute_delta':row[f'{ANALYSIS_FEATURES[0]}_absolute_delta'],'reporting_scale':codec_reporting_scale['tail excess'],'scaled_deviation':row[f'{ANALYSIS_FEATURES[0]}_absolute_delta']/codec_reporting_scale['tail excess']},
        {'condition':row.codec,'feature':'persistence','absolute_delta':row[f'{ANALYSIS_FEATURES[1]}_absolute_delta'],'reporting_scale':codec_reporting_scale['persistence'],'scaled_deviation':row[f'{ANALYSIS_FEATURES[1]}_absolute_delta']/codec_reporting_scale['persistence']},
        {'condition':row.codec,'feature':'decay','absolute_delta':row[f'{ANALYSIS_FEATURES[2]}_absolute_delta'],'reporting_scale':codec_reporting_scale['decay'],'scaled_deviation':decay_rel/codec_reporting_scale['decay']},
    ]
codec_plot=pd.DataFrame(codec_plot_rows)
fig,axes=plt.subplots(1,3,figsize=(14,4.2))
for feature in ANALYSIS_FEATURES[:3]:
    axes[0].plot(transform.condition,transform[feature]-transform.loc[transform.condition=='baseline',feature].iloc[0],'o-',label=feature.replace('qrev_',''))
axes[0].axhline(0,ls='--'); axes[0].set(ylabel='Change from baseline',title='Exact invariance/equivariance checks'); axes[0].tick_params(axis='x',rotation=28); axes[0].legend(fontsize=7)
for feature,local in resampling_plot.groupby('feature'):
    axes[1].bar(local.feature,local.normalized_deviation,label=feature)
axes[1].axhline(1,ls='--'); axes[1].set(ylabel='Absolute change / tolerance',title='Source-rate resampling'); axes[1].tick_params(axis='x',rotation=25)
for feature,local in codec_plot.groupby('feature'):
    axes[2].plot(local.condition,local.scaled_deviation,'o-',label=feature)
axes[2].axhline(1,ls='--'); axes[2].set(ylabel='Change / reporting scale',title='Codec characterization (not a pass threshold)'); axes[2].tick_params(axis='x',rotation=25); axes[2].legend(fontsize=8)
figure_source=pd.concat([
    transform.assign(section='exact_invariance'),
    resampling_plot.assign(section='resampling_tolerance'),
    codec_plot.assign(section='codec_reporting_scale'),
],ignore_index=True,sort=False)
fig_index.append(save_figure(fig,'C_transformation_contract',figure_source,'**Panel C. Transformation contract.** Conditional residual-tail features are required to be invariant to uniform gain, polarity, DC offset, and a common waveform/interval time shift. The SRMR feature pipeline is evaluated after the declared global DC-removal preprocessing step; the raw pinned upstream fixture remains a separate G2 implementation regression. Deterministic 48-to-16-kHz source-rate conversion is evaluated against preregistered feature-specific tolerances. FLAC, Opus, and AAC are characterized on an above-quantization fixture; the codec panel uses transparent reporting scales rather than acceptance thresholds because codec invariance is not assumed.','C','Do QREV estimators show their preregistered transformation behavior, and are codec effects characterized without presenting arbitrary thresholds as validation gates?'))
plt.close(fig)

In [ ]:
# G4 — RIR dose ordering, bounded persistence recovery, conditional decay recovery, and SRMR dose response
# Dry task with a very low independent floor, then convolve with controlled exponentially decaying RIRs.
dry, pr, sr = synthetic_task(tail_level=0, floor_level=2e-6, deterministic=True, seed=42)
rir_rows=[]
for rt60 in [0.0,.10,.20,.40,.60,.80]:
    reverberant=apply_rir(dry,rt60)
    result=ext(reverberant,pr,sr,recording_id=f'rt60_{rt60}').recording
    rir_rows.append({'rt60_sec':rt60,**{feature:result[feature] for feature in ANALYSIS_FEATURES[:3]},**{f'{feature}_status':result[f'{feature}_status'] for feature in ANALYSIS_FEATURES[:3]}})
rir_dose=pd.DataFrame(rir_rows)
valid_tail=rir_dose.loc[rir_dose.rt60_sec.le(.60)].dropna(subset=[ANALYSIS_FEATURES[0]])
valid_persist=rir_dose.loc[rir_dose.rt60_sec.le(.60)].dropna(subset=[ANALYSIS_FEATURES[1]])
check('G4','tail excess orders controlled RIR dose',stats.spearmanr(valid_tail.rt60_sec,valid_tail[ANALYSIS_FEATURES[0]]).statistic>=.8,stats.spearmanr(valid_tail.rt60_sec,valid_tail[ANALYSIS_FEATURES[0]]).statistic,'Spearman rho>=0.8')
check('G4','bounded persistence orders controlled RIR dose',stats.spearmanr(valid_persist.rt60_sec,valid_persist[ANALYSIS_FEATURES[1]]).statistic>=.8,stats.spearmanr(valid_persist.rt60_sec,valid_persist[ANALYSIS_FEATURES[1]]).statistic,'Spearman rho>=0.8')

persistence_rows=[]
for duration in [.06,.10,.16,.24,.32,.50,.65]:
    y,pp,ss=synthetic_task(tail_level=.012,plateau=duration,seed=33,deterministic=True)
    result=ext(y,pp,ss).recording
    persistence_rows.append({'injected_sec':duration,'measured_sec':result[ANALYSIS_FEATURES[1]],'status':result[f'{ANALYSIS_FEATURES[1]}_status'],'right_censored':result['qrev_persistence_recording_median_censored']})
persistence_dose=pd.DataFrame(persistence_rows)
finite_persistence=persistence_dose.dropna(subset=['measured_sec'])
check('G4','persistence recovers injected plateau duration within frame tolerance',np.nanmax(np.abs(finite_persistence.measured_sec.clip(upper=.6)-finite_persistence.injected_sec.clip(upper=.6)))<=.04,np.nanmax(np.abs(finite_persistence.measured_sec.clip(upper=.6)-finite_persistence.injected_sec.clip(upper=.6))),'<=0.04 s')

slope_rows=[]
for tau in [.06,.09,.12,.18,.25]:
    y,pp,ss=synthetic_task(tail_level=.02,tau=tau,floor_level=2e-6,seed=50,deterministic=False)
    measured=ext(y,pp,ss).recording[ANALYSIS_FEATURES[2]]
    expected=20/np.log(10)/tau
    slope_rows.append({'tau_sec':tau,'expected_db_per_sec':expected,'measured_db_per_sec':measured,'relative_error':abs(measured-expected)/expected})
decay_recovery=pd.DataFrame(slope_rows)
check('G4','conditional Theil–Sen decay recovery',decay_recovery.relative_error.max()<=.20,decay_recovery.relative_error.max(),'max relative error<=20%')

srmr_rir=pd.DataFrame()
if srmr_runtime_available:
    sample=loadmat(ROOT/'tests'/'fixtures'/'srmrpy'/'test.mat')['s'][:,0]
    rows=[]
    for rt60 in [0.0,.10,.20,.40,.60,.80]:
        score=feature_srmr(signal.fftconvolve(sample,synthetic_rir(rt60))[:len(sample)])
        rows.append({'rt60_sec':rt60,'qrev_srmr_norm':score})
    srmr_rir=pd.DataFrame(rows)
    rho=stats.spearmanr(srmr_rir.rt60_sec,srmr_rir.qrev_srmr_norm).statistic
    check('G4','pinned SRMR orders controlled RIR dose in expected direction',rho<=-.6,rho,'Spearman rho<=-0.6')

save_df(rir_dose,'qrev_v400_rir_dose')
save_df(persistence_dose,'qrev_v400_persistence_recovery')
save_df(decay_recovery,'qrev_v400_decay_recovery')
if len(srmr_rir): save_df(srmr_rir,'qrev_v400_srmr_rir_dose')

fig,axes=plt.subplots(2,2,figsize=(10.5,7.5))
within_contract=rir_dose.rt60_sec.le(.60)
stress_condition=~within_contract
axes[0,0].plot(rir_dose.loc[within_contract,'rt60_sec'],rir_dose.loc[within_contract,ANALYSIS_FEATURES[0]],'o-',label='dose range used for ordering')
axes[0,0].scatter(rir_dose.loc[stress_condition,'rt60_sec'],rir_dose.loc[stress_condition,ANALYSIS_FEATURES[0]],marker='x',s=70,label='late-floor overlap stress')
axes[0,0].set(xlabel='Controlled RIR RT60 parameter (s)',ylabel='Tail excess (dB)',title='Early residual magnitude'); axes[0,0].legend(fontsize=8)
axes[0,1].plot(rir_dose.loc[within_contract,'rt60_sec'],rir_dose.loc[within_contract,ANALYSIS_FEATURES[1]],'o-',label='dose range used for ordering')
axes[0,1].scatter(rir_dose.loc[stress_condition,'rt60_sec'],rir_dose.loc[stress_condition,ANALYSIS_FEATURES[1]],marker='x',s=70,label='right-censored stress')
axes[0,1].axhline(DEFAULT_PARAMETERS.persistence_horizon_ms/1000,ls='--',label='0.6-s horizon')
axes[0,1].set(xlabel='Controlled RIR RT60 parameter (s)',ylabel='Persistence (s)',title='Bounded observable persistence'); axes[0,1].legend(fontsize=8)
axes[1,0].plot(decay_recovery.expected_db_per_sec,decay_recovery.measured_db_per_sec,'o'); lim=[0,max(decay_recovery.expected_db_per_sec.max(),decay_recovery.measured_db_per_sec.max())]; axes[1,0].plot(lim,lim,'--'); axes[1,0].set(xlabel='Expected decay (dB/s)',ylabel='Measured decay (dB/s)',title='Conditional slope recovery')
if len(srmr_rir):
    axes[1,1].plot(srmr_rir.rt60_sec,srmr_rir.qrev_srmr_norm,'o-'); axes[1,1].set(xlabel='Controlled RIR RT60 parameter (s)',ylabel='Normalized-fast SRMR',title='Pinned SRMR RIR response')
else:
    axes[1,1].text(.5,.5,'SRMR runtime pending local pinned environment',ha='center',va='center'); axes[1,1].set_axis_off()
source=pd.concat([rir_dose.assign(section='boundary_rir',used_for_ordering=within_contract),decay_recovery.assign(section='decay_recovery'),srmr_rir.assign(section='srmr_rir')],ignore_index=True,sort=False)
fig_index.append(save_figure(fig,'A_construct_response',source,'**Panel A. Controlled construct response.** Tail excess and bounded persistence are evaluated across controlled RIR doses within the estimator operating range (0–0.6 s). The 0.8-s condition is shown separately as a late-floor/horizon stress condition and is excluded from the monotonic acceptance test. Conditional decay is compared with the known exponential-envelope slope; the pinned normalized-fast SRMR comparator is evaluated across the same RIR-dose ordering when its exact runtime is available. The RIR parameter is a simulation control, not an estimate returned by QREV.','A','Do the QREV measurements respond in the expected direction to controlled residual-tail and reverberation doses, and are operating-range failures shown rather than hidden?'))
plt.close(fig)

In [ ]:
# G5 — discriminant specificity and known non-identifiability
clean, pdry, sdry = synthetic_task(seed=91, deterministic=True)
rir_target = apply_rir(clean,.40)
condition_waveforms={
    'dry': clean,
    'RIR_target': rir_target,
    'stationary_pause_noise': add_post_offset(clean,pdry,'stationary',amp=.01),
    'late_floor_step': add_post_offset(clean,pdry,'late_floor_step',amp=.01),
    'breath_like': add_post_offset(clean,pdry,'breath',amp=.01),
    'discrete_echo': add_post_offset(clean,pdry,'echo',amp=.01),
    'rising_nondecay': add_post_offset(clean,pdry,'rising',amp=.01),
}
disc_rows=[]
for label,waveform in condition_waveforms.items():
    result=ext(waveform,pdry,sdry,recording_id=label).recording
    disc_rows.append({'condition':label,**{feature:result[feature] for feature in ANALYSIS_FEATURES[:3]},**{f'{feature}_status':result[f'{feature}_status'] for feature in ANALYSIS_FEATURES[:3]}})
discriminants=pd.DataFrame(disc_rows)
dry_tail=float(discriminants.loc[discriminants.condition=='dry',ANALYSIS_FEATURES[0]].iloc[0])
stationary_tail=float(discriminants.loc[discriminants.condition=='stationary_pause_noise',ANALYSIS_FEATURES[0]].iloc[0])
check('G5','stationary pause energy is largely removed by the independent local floor',abs(stationary_tail-dry_tail)<=3,stationary_tail-dry_tail,'absolute delta<=3 dB')
breath_tail=float(discriminants.loc[discriminants.condition=='breath_like',ANALYSIS_FEATURES[0]].iloc[0])
echo_tail=float(discriminants.loc[discriminants.condition=='discrete_echo',ANALYSIS_FEATURES[0]].iloc[0])
check('G5','breath-like post-offset energy is exposed as a positive confound',breath_tail>dry_tail+3,breath_tail,'> dry +3 dB')
check('G5','delayed discrete echo outside the 0-100-ms window is explicitly shown as outside retained-feature scope',abs(echo_tail-dry_tail)<=3,echo_tail-dry_tail,'absolute tail-excess delta<=3 dB; no discrete-echo detector claimed')
check('G5','rising/nondecaying trace is not encoded as zero downward decay',discriminants.loc[discriminants.condition=='rising_nondecay',f'{ANALYSIS_FEATURES[2]}_status'].iloc[0] != 'measured',discriminants.loc[discriminants.condition=='rising_nondecay',f'{ANALYSIS_FEATURES[2]}_status'].iloc[0],'not measured')
check('G5','late-floor instability changes eligibility/status rather than silently becoming a tail',discriminants.loc[discriminants.condition=='late_floor_step',f'{ANALYSIS_FEATURES[0]}_status'].iloc[0] in ['insufficient_support','measured'],discriminants.loc[discriminants.condition=='late_floor_step',f'{ANALYSIS_FEATURES[0]}_status'].iloc[0],'explicit measured/unavailable state with floor audit')

srmr_noise=pd.DataFrame()
if srmr_runtime_available:
    sample=loadmat(ROOT/'tests'/'fixtures'/'srmrpy'/'test.mat')['s'][:,0]
    rms=np.sqrt(np.mean(sample**2)); rows=[]; rng=np.random.default_rng(88)
    for snr_db in [np.inf,30,20,10]:
        noisy=sample.copy() if np.isinf(snr_db) else sample+rng.standard_normal(len(sample))*(rms/(10**(snr_db/20)))
        rows.append({'snr_db':'clean' if np.isinf(snr_db) else snr_db,'qrev_srmr_norm':feature_srmr(noisy)})
    srmr_noise=pd.DataFrame(rows)
    check('G5','SRMR additive-noise sensitivity characterized',len(srmr_noise)==4 and srmr_noise.qrev_srmr_norm.notna().all(),srmr_noise.to_dict('records'),'clean and 30/20/10-dB SNR conditions measured')

save_df(discriminants,'qrev_v400_discriminant_controls')
if len(srmr_noise): save_df(srmr_noise,'qrev_v400_srmr_noise_characterization')

fig,axes=plt.subplots(2,2,figsize=(11,7.5))
pretty_conditions={
    'dry':'Dry',
    'RIR_target':'RIR target',
    'stationary_pause_noise':'Stationary pause noise',
    'late_floor_step':'Late-floor change',
    'breath_like':'Breath-like',
    'discrete_echo':'Delayed echo',
    'rising_nondecay':'Rising/nondecay',
}
labels=[pretty_conditions.get(x,x) for x in discriminants.condition]
for axis,feature,title,ylabel in zip(
    axes.flat[:3],
    ANALYSIS_FEATURES[:3],
    ['Tail excess','Bounded persistence','Conditional downward decay'],
    ['Tail excess (dB)','Persistence (s)','Decay magnitude (dB/s)'],
):
    values=discriminants[feature].to_numpy(float)
    axis.bar(labels,np.nan_to_num(values,nan=0.0))
    for idx,value in enumerate(values):
        if not np.isfinite(value):
            axis.scatter(idx,0,marker='x',s=60)
            axis.annotate('NA',(idx,0),xytext=(0,6),textcoords='offset points',ha='center',fontsize=7)
    axis.tick_params(axis='x',rotation=35); axis.set(title=title,ylabel=ylabel)
if len(srmr_noise):
    axes.flat[3].plot(srmr_noise.snr_db.astype(str),srmr_noise.qrev_srmr_norm,'o-'); axes.flat[3].set(xlabel='Additive-noise condition',ylabel='Normalized-fast SRMR',title='Known SRMR noise sensitivity')
else:
    axes.flat[3].text(.5,.5,'SRMR noise characterization pending local runtime',ha='center',va='center'); axes.flat[3].set_axis_off()
source=pd.concat([discriminants.assign(section='boundary_discriminants'),srmr_noise.assign(section='srmr_noise')],ignore_index=True,sort=False)
fig_index.append(save_figure(fig,'B_discriminant_specificity',source,'**Panel B. Discriminant specificity.** The independent late-pause floor reduces stationary pause-energy effects, while breath-like residuals remain a deliberate positive confound. Delayed echo outside the first 100 ms is shown as outside the retained early-tail scope. Rising, nondecaying, or otherwise ineligible traces are marked unavailable rather than assigned zero. SRMR additive-noise sensitivity is characterized because SRMR is reverberation-sensitive but not reverberation-specific.','B','Can the retained measurements distinguish residual-tail manifestations from plausible competing mechanisms, and are unavailable states and unavoidable confounds made visible?'))
plt.close(fig)

In [ ]:
# G6 — support policy, censoring, boundary shifts, horizon/threshold, and floor-window sensitivity
support_rows=[]
y_support,p_support,s_support=synthetic_task(speech_count=3,tail_level=.01,plateau=.18,deterministic=True)
for minimum in [2,3,4]:
    prm=QREVParameters(minimum_tail_boundary_count=minimum,minimum_persistence_boundary_count=minimum,minimum_decay_boundary_count=minimum)
    result=ext(y_support,p_support,s_support,parameters=prm).recording
    support_rows.append({'minimum_boundaries':minimum,'tail_available':np.isfinite(result[ANALYSIS_FEATURES[0]]),'persistence_available':np.isfinite(result[ANALYSIS_FEATURES[1]]),'decay_available':np.isfinite(result[ANALYSIS_FEATURES[2]]),'tail_raw':result[f'{ANALYSIS_FEATURES[0]}_raw_estimate'],'tail_pause_support_sec':result['qrev_tail_valid_pause_support_sec']})
support_policy=pd.DataFrame(support_rows)
check('G6','raw estimates remain available when policy classifies analysis value unavailable',support_policy.tail_raw.notna().all(),support_policy.tail_raw.tolist(),'all finite raw estimates')

censor_rows=[]
y_cens,p_cens,s_cens=synthetic_task(tail_level=.015,plateau=.65,deterministic=True)
for horizon in [.4,.5,.6]:
    for threshold in [2.,3.,4.]:
        prm=QREVParameters(persistence_horizon_ms=horizon*1000,persistence_threshold_db=threshold,minimum_persistence_frame_count=max(40,int((horizon-.03)/.01)))
        result=ext(y_cens,p_cens,s_cens,parameters=prm).recording
        censor_rows.append({'horizon_sec':horizon,'threshold_db':threshold,'measured_sec':result[ANALYSIS_FEATURES[1]],'median_censored':result['qrev_persistence_recording_median_censored'],'status':result[f'{ANALYSIS_FEATURES[1]}_status']})
censoring=pd.DataFrame(censor_rows)
check('G6','persistence censoring is explicit at the observation horizon',censoring.median_censored.any() and censoring.status.eq('right_censored_at_horizon').any(),censoring[['horizon_sec','threshold_db','status']].to_dict('records'),'horizon values explicitly censored')

# Boundary perturbations are a sensitivity analysis, not alternative truth.
shift_rows=[]
for shift_ms in [-100,-50,0,50,100]:
    delta=shift_ms/1000
    shifted=[SpeechInterval(x.start_sec,x.end_sec+delta,x.interval_id,x.interval_index,x.view,x.profile) for x in primary]
    # keep intervals valid and ordered for this controlled fixture
    try:
        result=ext(base,shifted,strict,recording_id=f'shift_{shift_ms}').recording
        shift_rows.append({'offset_shift_ms':shift_ms,'status':'measured',**{feature:result[feature] for feature in ANALYSIS_FEATURES[:3]}})
    except Exception as exc:
        shift_rows.append({'offset_shift_ms':shift_ms,'status':f'error:{type(exc).__name__}:{exc}'})
boundary_sensitivity=pd.DataFrame(shift_rows)
check('G6','predeclared boundary-shift grid completed',len(boundary_sensitivity)==5 and boundary_sensitivity.status.eq('measured').all(),boundary_sensitivity.status.tolist(),'five measured shifts')

floor_rows=[]
for start_ms,end_ms in [(600,900),(700,1000),(750,1050)]:
    prm=QREVParameters(floor_start_ms=start_ms,floor_end_ms=end_ms)
    result=ext(base,primary,strict,parameters=prm).recording
    floor_rows.append({'floor_start_ms':start_ms,'floor_end_ms':end_ms,**{feature:result[feature] for feature in ANALYSIS_FEATURES[:3]},'tail_status':result[f'{ANALYSIS_FEATURES[0]}_status']})
floor_sensitivity=pd.DataFrame(floor_rows)
check('G6','independent nonoverlapping late-floor variants completed',(floor_sensitivity.floor_start_ms>=600).all() and len(floor_sensitivity)==3,floor_sensitivity[['floor_start_ms','floor_end_ms']].values.tolist(),'predeclared nonoverlapping late windows')

save_df(support_policy,'qrev_v400_support_policy_preflight')
save_df(censoring,'qrev_v400_censoring_preflight')
save_df(boundary_sensitivity,'qrev_v400_boundary_shift_preflight')
save_df(floor_sensitivity,'qrev_v400_floor_window_preflight')

In [ ]:
# Package tests, G1-G10 preflight status, figure index, and candidate manifest
if RUN_PACKAGE_TESTS:
    command=[sys.executable,'-m','pytest','tests reviewed/test_qrev_v400.py','-q','--disable-warnings']
    test_env=os.environ.copy()
    test_env['PYTHONPATH']=os.pathsep.join([
        str(ROOT/'src reviewed'),
        str(ROOT/'src'),
        test_env.get('PYTHONPATH',''),
    ])
    completed=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,env=test_env)
    check('G2','reviewed package tests',completed.returncode==0,(completed.stdout+'\n'+completed.stderr)[-1500:],'no failures')

all_checks=pd.DataFrame(checks)
save_df(all_checks,'qrev_v400_preflight_all_checks')

def gate_pass(gate):
    local=all_checks.loc[all_checks.gate.eq(gate)]
    return bool(len(local) and local.passed.all())

gates=pd.DataFrame([
    {'gate':'G1','status':'PASS' if gate_pass('G1') else 'FAIL','blocking':True,'summary':'construct, corrected primary-boundary provenance, pinned SRMR identity, no scalar'},
    {'gate':'G2','status':'PASS' if gate_pass('G2') else 'FAIL','blocking':True,'summary':'determinism, reconstruction, missing-not-zero, official SRMR runtime regression'},
    {'gate':'G3','status':'PASS' if gate_pass('G3') else 'FAIL','blocking':True,'summary':'gain/polarity/DC/time shift, source-rate resampling, codec and duration behavior'},
    {'gate':'G4','status':'PASS' if gate_pass('G4') else 'FAIL','blocking':True,'summary':'RIR dose ordering, bounded persistence, decay recovery, SRMR RIR response'},
    {'gate':'G5','status':'CONDITIONAL' if gate_pass('G5') else 'FAIL','blocking':True,'summary':'breath and echo remain explicit non-identifiable confounds; SRMR noise sensitivity exposed'},
    {'gate':'G6','status':'PREFLIGHT_PASS' if gate_pass('G6') else 'FAIL','blocking':True,'summary':'support, censoring, boundary, horizon and floor behavior; cohort policy pending'},
    {'gate':'G7','status':'PENDING','blocking':True,'summary':'corrected cohort extraction and empirical availability required'},
    {'gate':'G8','status':'PENDING','blocking':True,'summary':'participant-aware persistence, redundancy and support-threshold decision required'},
    {'gate':'G9','status':'N/A','blocking':False,'summary':'no retained discrete event detector'},
    {'gate':'G10','status':'PENDING','blocking':True,'summary':'feature decisions and immutable freeze prohibited before cohort audit'},
])
save_df(gates,'qrev_v400_gate_summary')
figure_index=pd.DataFrame(fig_index)
save_df(figure_index,'qrev_v400_preflight_figure_index')

preflight_gates=gates.loc[gates.gate.isin(['G1','G2','G3','G4','G5','G6'])]
preflight_pass=not preflight_gates.status.eq('FAIL').any()
manifest={
    'measurement_version':MEASUREMENT_VERSION,
    'family':'QREV',
    'candidate_only':True,
    'preflight_blocking_checks_pass':bool(preflight_pass),
    'cohort_extraction_completed':False,
    'freeze_allowed':False,
    'scientific_review_decision':SCIENTIFIC_REVIEW_DECISION,
    'boundary_contract':'primary_speech/primary natural offsets; strict_speech/primary support only',
    'legacy_qrev_v311_cohort_values_valid_for_reviewed_pipeline':False,
    'support_policy_status':'provisional_compare_2_3_4_on_corrected_cohort',
    'analysis_features':list(ANALYSIS_FEATURES),
    'srmr_variant':SRMR_VARIANT,
    'srmr_upstream_commit':SRMR_UPSTREAM_COMMIT,
    'gammatone_required_version':SRMR_GAMMATONE_VERSION,
    'srmr_runtime_available':bool(srmr_runtime_available),
    'global_dc_removal_enforced':True,
    'preflight_hotfix_revision':'dc-contract-hotfix-1',
    'parameter_sha256':PARAMETER_HASH,
    'implementation_sha256':IMPLEMENTATION_HASH,
    'figure_count':len(figure_index),
    'preflight_panels_complete':['A','B','C'] if len(figure_index)==3 else sorted(figure_index.panel.unique().tolist()),
    'family_scalar_constructed':False,
    'standalone_gate_allowed':False,
    'publish_and_freeze':PUBLISH_AND_FREEZE,
}
(MANIFESTS/'qrev_v400_preflight_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print(gates.to_string(index=False))
print(json.dumps(manifest,indent=2))
print('\nQREV v4.0.0 REVIEWED PREFLIGHT COMPLETE')